# Module 5 Stage 3 — sync verified files into the Drive project

Writes four files to `/content/drive/MyDrive/Fake_News_Detection`, **overwriting in place**
at their existing paths. No renames, no duplicates, no deletions of unrelated files.

The Drive connector available to Claude can only *create* files, never update or delete
one, so pushing these from outside would have produced a second `leakage_gate.py`
sitting next to the stale one. This notebook uses Colab's filesystem, which overwrites
correctly.

Replaced files are backed up to `backups/module5_stage3_<timestamp>/` first.

**This notebook does NOT run the corpus build.** It stops after verification.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
PAYLOAD_JSON = r'''{"src/fact_verification/leakage_gate.py": {"b64": "...", "sha256": "8f3608743399d5a2f1ceffd53ac9a2c4695df3a03a244169f661860b06851f64", "size": 16712}, "src/fact_verification/corpus_builder.py": {"b64": "...", "sha256": "9c9c3cbf16120ec85453cf56d577b772e261b4636542a1cf6179e6793f67331a", "size": 15308}, "tests/test_fact_verification.py": {"b64": "...", "sha256": "4a3e6a21f2b663e2ffb5101ab9ed45be1b9635def0e7ed0d3ddf4225547b47b2", "size": 31372}, "scripts/build_claim_corpus.py": {"b64": "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uCiIiIk1vZHVsZSA1IFN0YWdlIDMg4oCUIGJ1aWxkIHRoZSB1bmlmaWVkIGNsYWltL2V2aWRlbmNlIGNvcnB1cy4gQ09MQUIuCgpSZWFkcyB0aGUgZG93bmxvYWRlZCBYLUZBQ1QgYW5kIEZFVkVSIEpTT05MIGZpbGVzIGFuZCB3cml0ZXMgdGhlCmBjbGFpbXMuanNvbmxgIC8gYGV2aWRlbmNlLmpzb25sYCAvIGBjbGFpbV9zcGxpdHMuanNvbmAgYXJ0ZWZhY3RzIHRoZQpjb25maWcgYWxyZWFkeSBkZWNsYXJlcy4gTm8gcmV0cmlldmFsLCBubyBlbWJlZGRpbmdzLCBubyB0cmFpbmluZy4KCk9mZmljaWFsIHNwbGl0cyBhcmUgcHJlc2VydmVkIGV4YWN0bHkuIFRoZSBncm91cC1hd2FyZSA3MC8xNS8xNQpzcGxpdHRlciBpcyBOT1QgYXBwbGllZDogYm90aCBjb3Jwb3JhIHNoaXAgYSBwdWJsaXNoZWQgcHJvdG9jb2wsIGFuZApyZXNodWZmbGluZyBpdCB3b3VsZCBtYWtlIHJlc3VsdHMgaW5jb21wYXJhYmxlIHdpdGggcHJpb3Igd29yay4gVGhlCmxlYWthZ2UgZ2F0ZSBzdGlsbCBydW5zIGFnYWluc3QgdGhvc2Ugb2ZmaWNpYWwgc3BsaXRzIOKAlCBhIGZhaWx1cmUgdGhlcmUKaXMgYSBwcm9wZXJ0eSBvZiB0aGUgcHVibGlzaGVkIGRhdGFzZXQgYW5kIGlzIHJlcG9ydGVkIGFzIHN1Y2gsIG5ldmVyCnJlcGFpcmVkIGJ5IHNpbGVudGx5IHJlLXNwbGl0dGluZy4KClJlYWRzOiAgZGF0YXNldHMvbW9kdWxlNV9leHRlcm5hbC97eF9mYWN0LGZldmVyX2dvbGRfZXZpZGVuY2V9LyouanNvbmwKV3JpdGVzOiBkYXRhc2V0cy9mYWN0X3ZlcmlmaWNhdGlvbi97Y2xhaW1zLGV2aWRlbmNlfS5qc29ubCwKICAgICAgICBjbGFpbV9zcGxpdHMuanNvbiwgYW5kIG91dHB1dHMvbW9kdWxlNS9jb3JwdXNfYnVpbGRfcmVwb3J0Lntqc29uLG1kfQoKTmV2ZXIgdG91Y2hlcyBkYXRhc2V0cy9wcm9jZXNzZWQvLCBlbWJlZGRpbmdzLywgb3V0cHV0cy9tb2R1bGUzLywKb3V0cHV0cy9tb2R1bGU0LyBvciBtb2RlbHMvbW9kdWxlNC8uCgpVc2FnZToKICAgIHB5dGhvbiBzY3JpcHRzL2J1aWxkX2NsYWltX2NvcnB1cy5weQogICAgcHl0aG9uIHNjcmlwdHMvYnVpbGRfY2xhaW1fY29ycHVzLnB5IC0tb25seSB4X2ZhY3QKICAgIHB5dGhvbiBzY3JpcHRzL2J1aWxkX2NsYWltX2NvcnB1cy5weSAtLXNraXAtbGVha2FnZS1jaGVjayAgICMgbGFyZ2UgY29ycG9yYQoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQganNvbgppbXBvcnQgc3lzCmltcG9ydCB0aW1lCmZyb20gZGF0ZXRpbWUgaW1wb3J0IGRhdGV0aW1lLCB0aW1lem9uZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB0eXBpbmcgaW1wb3J0IEFueQoKUFJPSkVDVF9ST09UID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudHNbMV0Kc3lzLnBhdGguaW5zZXJ0KDAsIHN0cihQUk9KRUNUX1JPT1QpKQoKZnJvbSBzcmMuZmFjdF92ZXJpZmljYXRpb24uY29ycHVzX2J1aWxkZXIgaW1wb3J0ICggICMgbm9xYTogRTQwMgogICAgYnVpbGRfY29ycHVzX2Zyb21fZmlsZXMsCiAgICBzcGxpdHNfZnJvbV9jbGFpbXMsCiAgICB3cml0ZV9jb3JwdXMsCikKZnJvbSBzcmMuZmFjdF92ZXJpZmljYXRpb24ubGVha2FnZV9nYXRlIGltcG9ydCBncm91cF9jbGFpbXMsIHZlcmlmeV9ub19sZWFrYWdlICAjIG5vcWE6IEU0MDIKZnJvbSBzcmMudXRpbHMuY29uZmlnX2xvYWRlciBpbXBvcnQgbG9hZF95YW1sICAjIG5vcWE6IEU0MDIKZnJvbSBzcmMudXRpbHMubG9nZ2VyIGltcG9ydCBnZXRfbG9nZ2VyICAjIG5vcWE6IEU0MDIKCmxvZ2dlciA9IGdldF9sb2dnZXIoIm1vZHVsZTUuYnVpbGRfY29ycHVzIikKCkZST1pFTiA9IFsiZGF0YXNldHMvcHJvY2Vzc2VkIiwgImVtYmVkZGluZ3MiLCAib3V0cHV0cy9tb2R1bGUzIiwgIm91dHB1dHMvbW9kdWxlNCIsICJtb2RlbHMvbW9kdWxlNCJdCgojIChkaXJlY3RvcnksIGRhdGFzZXQgdGFnLCB7ZmlsZW5hbWUgc3RlbSBzdWZmaXggLT4gb2ZmaWNpYWwgc3BsaXQgbmFtZX0pCkNPUlBVU19MQVlPVVQgPSB7CiAgICAieF9mYWN0IjogKCJ4X2ZhY3QiLCAieF9mYWN0IiwgeyJ0cmFpbiI6ICJ0cmFpbiIsICJkZXYiOiAiZGV2IiwgInRlc3QiOiAidGVzdCJ9KSwKICAgICJmZXZlciI6ICgiZmV2ZXJfZ29sZF9ldmlkZW5jZSIsICJmZXZlciIsIHsidHJhaW4iOiAidHJhaW4iLCAidmFsaWRhdGlvbiI6ICJ2YWxpZGF0aW9uIiwgInRlc3QiOiAidGVzdCJ9KSwKfQoKCmRlZiBmcm96ZW5fc2lnbmF0dXJlKHJvb3Q6IFBhdGgpIC0+IGRpY3Rbc3RyLCBBbnldOgogICAgc2lnOiBkaWN0W3N0ciwgQW55XSA9IHt9CiAgICBmb3IgcmVsIGluIEZST1pFTjoKICAgICAgICBwID0gcm9vdCAvIHJlbAogICAgICAgIGlmIG5vdCBwLmV4aXN0cygpOgogICAgICAgICAgICBzaWdbcmVsXSA9IE5vbmUKICAgICAgICAgICAgY29udGludWUKICAgICAgICBmaWxlcyA9IFtmIGZvciBmIGluIHAucmdsb2IoIioiKSBpZiBmLmlzX2ZpbGUoKV0KICAgICAgICBzaWdbcmVsXSA9IHsibl9maWxlcyI6IGxlbihmaWxlcyksICJ0b3RhbF9ieXRlcyI6IHN1bShmLnN0YXQoKS5zdF9zaXplIGZvciBmIGluIGZpbGVzKX0KICAgIHJldHVybiBzaWcKCgpkZWYgZGlzY292ZXJfZmlsZXMoY29ycHVzX2RpcjogUGF0aCwgc3BsaXRfbWFwOiBkaWN0W3N0ciwgc3RyXSkgLT4gbGlzdFt0dXBsZVtQYXRoLCBzdHJdXToKICAgICIiIk1hdGNoIGVhY2ggSlNPTkwgdG8gaXRzIG9mZmljaWFsIHNwbGl0IGJ5IGZpbGVuYW1lIHN1ZmZpeC4iIiIKICAgIGZvdW5kOiBsaXN0W3R1cGxlW1BhdGgsIHN0cl1dID0gW10KICAgIGZvciBmcCBpbiBzb3J0ZWQoY29ycHVzX2Rpci5nbG9iKCIqLmpzb25sIikpOgogICAgICAgIHN0ZW0gPSBmcC5zdGVtLmxvd2VyKCkKICAgICAgICBmb3Igc3VmZml4LCBvZmZpY2lhbCBpbiBzcGxpdF9tYXAuaXRlbXMoKToKICAgICAgICAgICAgaWYgc3RlbS5lbmRzd2l0aChmIl97c3VmZml4fSIpOgogICAgICAgICAgICAgICAgZm91bmQuYXBwZW5kKChmcCwgb2ZmaWNpYWwpKQogICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICBlbHNlOgogICAgICAgICAgICBsb2dnZXIud2FybmluZygiVW5yZWNvZ25pc2VkIHNwbGl0IGZvciAlcyDigJQgc2tpcHBlZCAoZXhwZWN0ZWQgc3VmZml4ZXM6ICVzKSIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgIGZwLm5hbWUsIHNvcnRlZChzcGxpdF9tYXApKQogICAgcmV0dXJuIGZvdW5kCgoKZGVmIG1haW4oKSAtPiBOb25lOgogICAgcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPSJNb2R1bGUgNSBTdGFnZSAzOiBidWlsZCB0aGUgdW5pZmllZCBjbGFpbSBjb3JwdXMuIikKICAgIHAuYWRkX2FyZ3VtZW50KCItLWNvbmZpZyIsIGRlZmF1bHQ9Tm9uZSkKICAgIHAuYWRkX2FyZ3VtZW50KCItLWRhdGEtZGlyIiwgZGVmYXVsdD0iZGF0YXNldHMvbW9kdWxlNV9leHRlcm5hbCIpCiAgICBwLmFkZF9hcmd1bWVudCgiLS1vbmx5IiwgY2hvaWNlcz1bInhfZmFjdCIsICJmZXZlciJdLCBkZWZhdWx0PU5vbmUpCiAgICBwLmFkZF9hcmd1bWVudCgiLS1za2lwLWxlYWthZ2UtY2hlY2siLCBhY3Rpb249InN0b3JlX3RydWUiLAogICAgICAgICAgICAgICAgICAgaGVscD0iU2tpcCB0aGUgY3Jvc3Mtc3BsaXQgbmVhci1kdXBsaWNhdGUgc2NhbiAoZXhwZW5zaXZlIG9uIEZFVkVSKS4iKQogICAgYXJncyA9IHAucGFyc2VfYXJncygpCgogICAgcm9vdCA9IFBST0pFQ1RfUk9PVAogICAgY2ZnID0gbG9hZF95YW1sKFBhdGgoYXJncy5jb25maWcpIGlmIGFyZ3MuY29uZmlnIGVsc2Ugcm9vdCAvICJjb25maWdzIiAvICJtb2R1bGU1X2ZhY3RfdmVyaWZpY2F0aW9uLnlhbWwiKQogICAgZGF0YV9kaXIgPSByb290IC8gYXJncy5kYXRhX2RpcgogICAgb3V0X3JlcG9ydHMgPSByb290IC8gY2ZnWyJvdXRwdXQiXVsicmVwb3J0c19kaXIiXQogICAgb3V0X2NvcnB1c19yb290ID0gcm9vdCAvIGNmZ1sib3V0cHV0Il1bImNvcnB1c19kaXIiXQogICAgb3V0X3JlcG9ydHMubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQoKICAgIGJlZm9yZSA9IGZyb3plbl9zaWduYXR1cmUocm9vdCkKICAgIGxrID0gY2ZnWyJsZWFrYWdlIl0KCiAgICByZXBvcnQ6IGRpY3Rbc3RyLCBBbnldID0gewogICAgICAgICJtb2R1bGUiOiAiTW9kdWxlIDUgU3RhZ2UgMyDigJQgY2xhaW0vZXZpZGVuY2UgY29ycHVzIGNvbnN0cnVjdGlvbiIsCiAgICAgICAgImdlbmVyYXRlZF9hdCI6IGRhdGV0aW1lLm5vdyh0aW1lem9uZS51dGMpLmlzb2Zvcm1hdCgpLAogICAgICAgICJzcGxpdF9wb2xpY3kiOiAiUFJFU0VSVkVfT0ZGSUNJQUwiLAogICAgICAgICJzcGxpdF9wb2xpY3lfbm90ZSI6ICgiWC1GQUNUIGFuZCBGRVZFUiBib3RoIHNoaXAgb2ZmaWNpYWwgc3BsaXRzOyB0aGVzZSBhcmUgcHJlc2VydmVkICIKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgInZlcmJhdGltLiBUaGUgZ3JvdXAtYXdhcmUgNzAvMTUvMTUgc3BsaXR0ZXIgaXMgcmVzZXJ2ZWQgZm9yIGNvcnBvcmEgIgogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAid2l0aG91dCBhIHVzYWJsZSBvZmZpY2lhbCBzcGxpdCBhbmQgd2FzIE5PVCBhcHBsaWVkLiIpLAogICAgICAgICJjb3Jwb3JhIjoge30sCiAgICAgICAgIndhcm5pbmdzIjogW10sCiAgICB9CgogICAgdGFyZ2V0cyA9IFthcmdzLm9ubHldIGlmIGFyZ3Mub25seSBlbHNlIFsieF9mYWN0IiwgImZldmVyIl0KICAgIGZvciBrZXkgaW4gdGFyZ2V0czoKICAgICAgICBzdWJkaXIsIGRhdGFzZXQsIHNwbGl0X21hcCA9IENPUlBVU19MQVlPVVRba2V5XQogICAgICAgIGNvcnB1c19kaXIgPSBkYXRhX2RpciAvIHN1YmRpcgogICAgICAgIGlmIG5vdCBjb3JwdXNfZGlyLmV4aXN0cygpOgogICAgICAgICAgICByZXBvcnRbIndhcm5pbmdzIl0uYXBwZW5kKGYie2RhdGFzZXR9OiB7Y29ycHVzX2Rpcn0gbm90IGZvdW5kIOKAlCBydW4gdGhlIGRvd25sb2FkIHNjcmlwdCBmaXJzdCIpCiAgICAgICAgICAgIGNvbnRpbnVlCgogICAgICAgIGZpbGVzID0gZGlzY292ZXJfZmlsZXMoY29ycHVzX2Rpciwgc3BsaXRfbWFwKQogICAgICAgIGlmIG5vdCBmaWxlczoKICAgICAgICAgICAgcmVwb3J0WyJ3YXJuaW5ncyJdLmFwcGVuZChmIntkYXRhc2V0fTogbm8gcmVjb2duaXNlZCBzcGxpdCBmaWxlcyBpbiB7Y29ycHVzX2Rpcn0iKQogICAgICAgICAgICBjb250aW51ZQoKICAgICAgICBsb2dnZXIuaW5mbygiWyVzXSBidWlsZGluZyBmcm9tICVkIGZpbGUocykuLi4iLCBkYXRhc2V0LCBsZW4oZmlsZXMpKQogICAgICAgIHQwID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgICAgIGNsYWltcywgZXZpZGVuY2VfcG9vbCwgc3RhdHMgPSBidWlsZF9jb3JwdXNfZnJvbV9maWxlcyhmaWxlcywgZGF0YXNldCwgY2ZnKQogICAgICAgIGJ1aWxkX3NlY29uZHMgPSByb3VuZCh0aW1lLnBlcmZfY291bnRlcigpIC0gdDAsIDIpCgogICAgICAgIGlmIG5vdCBzdGF0c1sicm93X2FjY291bnRpbmdfYmFsYW5jZXMiXToKICAgICAgICAgICAgcmVwb3J0WyJ3YXJuaW5ncyJdLmFwcGVuZCgKICAgICAgICAgICAgICAgIGYie2RhdGFzZXR9OiBST1cgQUNDT1VOVElORyBET0VTIE5PVCBCQUxBTkNFIOKAlCAiCiAgICAgICAgICAgICAgICBmIntzdGF0c1snbl9pbnB1dF9yb3dzJ119IGluIHZzIHtzdGF0c1snbl9jbGFpbXNfYnVpbHQnXX0gYnVpbHQgKyAiCiAgICAgICAgICAgICAgICBmIntzdGF0c1snbl9kcm9wcGVkX25vX2NsYWltX3RleHQnXX0gKyB7c3RhdHNbJ25fZHJvcHBlZF9iZWxvd19taW5fY2hhcnMnXX0gZHJvcHBlZCIpCgogICAgICAgIHNwbGl0cyA9IHNwbGl0c19mcm9tX2NsYWltcyhjbGFpbXMpCgogICAgICAgIGxvZ2dlci5pbmZvKCJbJXNdIGdyb3VwaW5nICVkIGNsYWltcyAodGhyZXNob2xkPSUuMmYsIG5ncmFtPSVkKS4uLiIsCiAgICAgICAgICAgICAgICAgICAgZGF0YXNldCwgbGVuKGNsYWltcyksIGxrWyJuZWFyX2R1cGxpY2F0ZV90aHJlc2hvbGQiXSwgbGtbIm5lYXJfZHVwbGljYXRlX25ncmFtIl0pCiAgICAgICAgdDEgPSB0aW1lLnBlcmZfY291bnRlcigpCiAgICAgICAgZ3JvdXBpbmcgPSBncm91cF9jbGFpbXMoY2xhaW1zLCBuZWFyX2R1cGxpY2F0ZT1UcnVlLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG5ncmFtPWxrWyJuZWFyX2R1cGxpY2F0ZV9uZ3JhbSJdLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHRocmVzaG9sZD1sa1sibmVhcl9kdXBsaWNhdGVfdGhyZXNob2xkIl0pCiAgICAgICAgZ3JvdXBfc2Vjb25kcyA9IHJvdW5kKHRpbWUucGVyZl9jb3VudGVyKCkgLSB0MSwgMikKCiAgICAgICAgbGVha2FnZSA9IE5vbmUKICAgICAgICBpZiBub3QgYXJncy5za2lwX2xlYWthZ2VfY2hlY2s6CiAgICAgICAgICAgIGxvZ2dlci5pbmZvKCJbJXNdIHJ1bm5pbmcgbGVha2FnZSBnYXRlIG9uIE9GRklDSUFMIHNwbGl0cy4uLiIsIGRhdGFzZXQpCiAgICAgICAgICAgIHQyID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgICAgICAgICBsZWFrYWdlID0gdmVyaWZ5X25vX2xlYWthZ2UoCiAgICAgICAgICAgICAgICBjbGFpbXMsIHNwbGl0cywgZ3JvdXBpbmcuZ3JvdXBfaWRzLAogICAgICAgICAgICAgICAgbmVhcl9kdXBsaWNhdGVfdGhyZXNob2xkPWxrWyJuZWFyX2R1cGxpY2F0ZV90aHJlc2hvbGQiXSwKICAgICAgICAgICAgICAgIG5ncmFtPWxrWyJuZWFyX2R1cGxpY2F0ZV9uZ3JhbSJdKQogICAgICAgICAgICBsZWFrYWdlWyJzZWNvbmRzIl0gPSByb3VuZCh0aW1lLnBlcmZfY291bnRlcigpIC0gdDIsIDIpCiAgICAgICAgICAgIGxlYWthZ2VbImludGVycHJldGF0aW9uIl0gPSAoCiAgICAgICAgICAgICAgICAiTWVhc3VyZWQgb24gdGhlIGNvcnB1cydzIE9XTiBvZmZpY2lhbCBzcGxpdHMuIEEgZmFpbHVyZSBoZXJlIGRlc2NyaWJlcyB0aGUgIgogICAgICAgICAgICAgICAgInB1Ymxpc2hlZCBkYXRhc2V0LCBub3Qgb3VyIHNwbGl0dGluZy4gUmVwb3J0IGl0OyBkbyBub3Qgc2lsZW50bHkgcmUtc3BsaXQuIikKICAgICAgICAgICAgaWYgbm90IGxlYWthZ2VbInBhc3NlZCJdOgogICAgICAgICAgICAgICAgcmVwb3J0WyJ3YXJuaW5ncyJdLmFwcGVuZCgKICAgICAgICAgICAgICAgICAgICBmIntkYXRhc2V0fTogbGVha2FnZSBnYXRlIEZBSUxFRCBvbiBvZmZpY2lhbCBzcGxpdHMg4oCUIHNlZSBjb3JwdXNfYnVpbGRfcmVwb3J0Lmpzb24iKQoKICAgICAgICBvdXRfZGlyID0gb3V0X2NvcnB1c19yb290IC8gZGF0YXNldAogICAgICAgIHBhdGhzID0gd3JpdGVfY29ycHVzKGNsYWltcywgZXZpZGVuY2VfcG9vbCwgc3BsaXRzLCBncm91cGluZy5ncm91cF9pZHMsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgb3V0X2RpciwgY2ZnWyJvdXRwdXQiXVsiY2xhaW1zX2ZpbGVuYW1lIl0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgc3BsaXRzX2ZpbGVuYW1lPWNmZ1sib3V0cHV0Il1bInNwbGl0c19maWxlbmFtZSJdKQoKICAgICAgICByZXBvcnRbImNvcnBvcmEiXVtkYXRhc2V0XSA9IHsKICAgICAgICAgICAgInNvdXJjZV9maWxlcyI6IFtmIntmcC5uYW1lfSAtPiB7c3B9IiBmb3IgZnAsIHNwIGluIGZpbGVzXSwKICAgICAgICAgICAgImJ1aWxkX3NlY29uZHMiOiBidWlsZF9zZWNvbmRzLAogICAgICAgICAgICAiZ3JvdXBpbmdfc2Vjb25kcyI6IGdyb3VwX3NlY29uZHMsCiAgICAgICAgICAgICJzdGF0aXN0aWNzIjogc3RhdHMsCiAgICAgICAgICAgICJzcGxpdHMiOiB7azogbGVuKHYpIGZvciBrLCB2IGluIHNwbGl0cy5pdGVtcygpfSwKICAgICAgICAgICAgImdyb3VwaW5nIjogZ3JvdXBpbmcuc3VtbWFyeSwKICAgICAgICAgICAgImxlYWthZ2UiOiBsZWFrYWdlLAogICAgICAgICAgICAib3V0cHV0X2ZpbGVzIjogcGF0aHMsCiAgICAgICAgfQogICAgICAgIGxvZ2dlci5pbmZvKCJbJXNdICVkIGNsYWltcywgJWQgdW5pcXVlIGV2aWRlbmNlIGl0ZW1zLCAlZCBncm91cHMgLT4gJXMiLAogICAgICAgICAgICAgICAgICAgIGRhdGFzZXQsIGxlbihjbGFpbXMpLCBsZW4oZXZpZGVuY2VfcG9vbCksIGdyb3VwaW5nLm5fZ3JvdXBzLCBvdXRfZGlyKQoKICAgIGFmdGVyID0gZnJvemVuX3NpZ25hdHVyZShyb290KQogICAgY2hhbmdlZCA9IHtrOiB7ImJlZm9yZSI6IGJlZm9yZS5nZXQoayksICJhZnRlciI6IGFmdGVyLmdldChrKX0KICAgICAgICAgICAgICAgZm9yIGsgaW4gYWZ0ZXIgaWYgYmVmb3JlLmdldChrKSAhPSBhZnRlci5nZXQoayl9CiAgICByZXBvcnRbIm1vZHVsZXNfMV80X2ludGVncml0eSJdID0geyJ1bmNoYW5nZWQiOiBub3QgY2hhbmdlZCwgImNoYW5nZWQiOiBjaGFuZ2VkfQogICAgaWYgY2hhbmdlZDoKICAgICAgICByZXBvcnRbIndhcm5pbmdzIl0uYXBwZW5kKGYiRlJPWkVOIE1PRFVMRVMgQ0hBTkdFRDoge3NvcnRlZChjaGFuZ2VkKX0iKQoKICAgIChvdXRfcmVwb3J0cyAvICJjb3JwdXNfYnVpbGRfcmVwb3J0Lmpzb24iKS53cml0ZV90ZXh0KAogICAgICAgIGpzb24uZHVtcHMocmVwb3J0LCBpbmRlbnQ9MiwgZW5zdXJlX2FzY2lpPUZhbHNlLCBkZWZhdWx0PXN0ciksIGVuY29kaW5nPSJ1dGYtOCIpCiAgICBfd3JpdGVfbWQocmVwb3J0LCBvdXRfcmVwb3J0cyAvICJjb3JwdXNfYnVpbGRfcmVwb3J0Lm1kIikKCiAgICBwcmludCgiXG5NT0RVTEUgNSBTVEFHRSAzIOKAlCBDTEFJTSBDT1JQVVMgQlVJTEQiKQogICAgcHJpbnQoIi0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0iKQogICAgZm9yIG5hbWUsIHIgaW4gcmVwb3J0WyJjb3Jwb3JhIl0uaXRlbXMoKToKICAgICAgICBzID0gclsic3RhdGlzdGljcyJdCiAgICAgICAgcHJpbnQoZiJcbntuYW1lfToiKQogICAgICAgIHByaW50KGYiICBpbnB1dCByb3dzICAgICAgICA6IHtzWyduX2lucHV0X3Jvd3MnXX0iKQogICAgICAgIHByaW50KGYiICBjbGFpbXMgYnVpbHQgICAgICA6IHtzWyduX2NsYWltc19idWlsdCddfSAgIgogICAgICAgICAgICAgIGYiKGRyb3BwZWQ6IHtzWyduX2Ryb3BwZWRfbm9fY2xhaW1fdGV4dCddfSBuby10ZXh0LCAiCiAgICAgICAgICAgICAgZiJ7c1snbl9kcm9wcGVkX2JlbG93X21pbl9jaGFycyddfSBiZWxvdy1taW4pIikKICAgICAgICBwcmludChmIiAgcm93IGFjY291bnRpbmcgICAgOiB7J0JBTEFOQ0VTJyBpZiBzWydyb3dfYWNjb3VudGluZ19iYWxhbmNlcyddIGVsc2UgJ01JU01BVENIJ30iKQogICAgICAgIHByaW50KGYiICBvZmZpY2lhbCBzcGxpdHMgICA6IHtyWydzcGxpdHMnXX0iKQogICAgICAgIHByaW50KGYiICBsYW5ndWFnZXMgICAgICAgICA6IHtzWydieV9sYW5ndWFnZSddfSIpCiAgICAgICAgcHJpbnQoZiIgIGxhYmVscyAgICAgICAgICAgIDoge3NbJ2J5X2xhYmVsJ119IikKICAgICAgICBwcmludChmIiAgZXZpZGVuY2UgaXRlbXMgICAgOiB7c1snbl91bmlxdWVfZXZpZGVuY2VfaXRlbXMnXX0gdW5pcXVlICIKICAgICAgICAgICAgICBmIih7c1snbl9ldmlkZW5jZV9tZW50aW9ucyddfSBtZW50aW9ucykiKQogICAgICAgIHByaW50KGYiICBjbGFpbXMgdy9vIGV2aWRlbmNlOiB7c1snbl9jbGFpbXNfd2l0aF9ub19ldmlkZW5jZSddfSIpCiAgICAgICAgcHJpbnQoZiIgIGdyb3VwcyAgICAgICAgICAgIDoge3JbJ2dyb3VwaW5nJ11bJ25fZ3JvdXBzJ119ICIKICAgICAgICAgICAgICBmIihsYXJnZXN0IHtyWydncm91cGluZyddWydsYXJnZXN0X2dyb3VwX3NpemUnXX0sICIKICAgICAgICAgICAgICBmIntyWydncm91cGluZyddWyduZWFyX2R1cGxpY2F0ZV9wYWlycyddfSBuZWFyLWR1cCBwYWlycykiKQogICAgICAgIGlmIHJbImxlYWthZ2UiXToKICAgICAgICAgICAgcHJpbnQoZiIgIGxlYWthZ2UgZ2F0ZSAgICAgIDogeydQQVNTJyBpZiByWydsZWFrYWdlJ11bJ3Bhc3NlZCddIGVsc2UgJ0ZBSUwnfSAiCiAgICAgICAgICAgICAgICAgIGYiKHtyWydsZWFrYWdlJ11bJ25fcGFzc2VkJ119L3tyWydsZWFrYWdlJ11bJ25fY2hlY2tzJ119LCB7clsnbGVha2FnZSddWydzZWNvbmRzJ119cykiKQogICAgcHJpbnQoZiJcbk1vZHVsZXMgMS00IHVuY2hhbmdlZDoge3JlcG9ydFsnbW9kdWxlc18xXzRfaW50ZWdyaXR5J11bJ3VuY2hhbmdlZCddfSIpCiAgICBpZiByZXBvcnRbIndhcm5pbmdzIl06CiAgICAgICAgcHJpbnQoZiJcbldBUk5JTkdTICh7bGVuKHJlcG9ydFsnd2FybmluZ3MnXSl9KToiKQogICAgICAgIGZvciB3IGluIHJlcG9ydFsid2FybmluZ3MiXToKICAgICAgICAgICAgcHJpbnQoZiIgIC0ge3d9IikKICAgIHByaW50KGYiXG5Xcm90ZToge291dF9yZXBvcnRzLydjb3JwdXNfYnVpbGRfcmVwb3J0Lmpzb24nfSIpCiAgICBwcmludChmIldyb3RlOiB7b3V0X3JlcG9ydHMvJ2NvcnB1c19idWlsZF9yZXBvcnQubWQnfSIpCgogICAgZmFpbGVkID0gKG5vdCByZXBvcnRbIm1vZHVsZXNfMV80X2ludGVncml0eSJdWyJ1bmNoYW5nZWQiXQogICAgICAgICAgICAgIG9yIGFueShub3Qgclsic3RhdGlzdGljcyJdWyJyb3dfYWNjb3VudGluZ19iYWxhbmNlcyJdIGZvciByIGluIHJlcG9ydFsiY29ycG9yYSJdLnZhbHVlcygpKSkKICAgIHN5cy5leGl0KDEgaWYgZmFpbGVkIGVsc2UgMCkKCgpkZWYgX3dyaXRlX21kKHJlcG9ydDogZGljdFtzdHIsIEFueV0sIHBhdGg6IFBhdGgpIC0+IE5vbmU6CiAgICBMID0gWyIjIE1vZHVsZSA1IFN0YWdlIDMg4oCUIENsYWltL0V2aWRlbmNlIENvcnB1cyBCdWlsZCIsICIiLAogICAgICAgICBmIkdlbmVyYXRlZDoge3JlcG9ydFsnZ2VuZXJhdGVkX2F0J119IiwgIiIsCiAgICAgICAgIGYiKipTcGxpdCBwb2xpY3k6IHtyZXBvcnRbJ3NwbGl0X3BvbGljeSddfSoqIiwgIiIsCiAgICAgICAgIHJlcG9ydFsic3BsaXRfcG9saWN5X25vdGUiXSwgIiJdCiAgICBmb3IgbmFtZSwgciBpbiByZXBvcnRbImNvcnBvcmEiXS5pdGVtcygpOgogICAgICAgIHMgPSByWyJzdGF0aXN0aWNzIl0KICAgICAgICBMICs9IFtmIiMjIHtuYW1lfSIsICIiLAogICAgICAgICAgICAgICJ8IHwgfCIsICJ8LS0tfC0tLTp8IiwKICAgICAgICAgICAgICBmInwgSW5wdXQgcm93cyB8IHtzWyduX2lucHV0X3Jvd3MnXX0gfCIsCiAgICAgICAgICAgICAgZiJ8IENsYWltcyBidWlsdCB8IHtzWyduX2NsYWltc19idWlsdCddfSB8IiwKICAgICAgICAgICAgICBmInwgRHJvcHBlZCAobm8gY2xhaW0gdGV4dCkgfCB7c1snbl9kcm9wcGVkX25vX2NsYWltX3RleHQnXX0gfCIsCiAgICAgICAgICAgICAgZiJ8IERyb3BwZWQgKGJlbG93IG1pbiBjaGFycykgfCB7c1snbl9kcm9wcGVkX2JlbG93X21pbl9jaGFycyddfSB8IiwKICAgICAgICAgICAgICBmInwgUm93IGFjY291bnRpbmcgYmFsYW5jZXMgfCB7c1sncm93X2FjY291bnRpbmdfYmFsYW5jZXMnXX0gfCIsCiAgICAgICAgICAgICAgZiJ8IFVuaXF1ZSBldmlkZW5jZSBpdGVtcyB8IHtzWyduX3VuaXF1ZV9ldmlkZW5jZV9pdGVtcyddfSB8IiwKICAgICAgICAgICAgICBmInwgRXZpZGVuY2UgbWVudGlvbnMgfCB7c1snbl9ldmlkZW5jZV9tZW50aW9ucyddfSB8IiwKICAgICAgICAgICAgICBmInwgQ2xhaW1zIHdpdGhvdXQgZXZpZGVuY2UgfCB7c1snbl9jbGFpbXNfd2l0aF9ub19ldmlkZW5jZSddfSB8IiwKICAgICAgICAgICAgICBmInwgQ2xhaW0gZ3JvdXBzIHwge3JbJ2dyb3VwaW5nJ11bJ25fZ3JvdXBzJ119IHwiLAogICAgICAgICAgICAgIGYifCBMYXJnZXN0IGdyb3VwIHwge3JbJ2dyb3VwaW5nJ11bJ2xhcmdlc3RfZ3JvdXBfc2l6ZSddfSB8IiwgIiIsCiAgICAgICAgICAgICAgZiJPZmZpY2lhbCBzcGxpdHM6IGB7clsnc3BsaXRzJ119YCIsICIiLAogICAgICAgICAgICAgIGYiTGFuZ3VhZ2VzOiBge3NbJ2J5X2xhbmd1YWdlJ119YCIsICIiLAogICAgICAgICAgICAgIGYiTGFiZWxzOiBge3NbJ2J5X2xhYmVsJ119YCIsICIiLAogICAgICAgICAgICAgIGYiRXZpZGVuY2UgZXh0cmFjdGlvbjogYHtzWydldmlkZW5jZV9leHRyYWN0aW9uX21ldGhvZHMnXX1gIiwgIiJdCiAgICAgICAgaWYgclsibGVha2FnZSJdOgogICAgICAgICAgICBMICs9IFtmIiMjIyBMZWFrYWdlIGdhdGUgb24gb2ZmaWNpYWwgc3BsaXRzOiAiCiAgICAgICAgICAgICAgICAgIGYiKip7J1BBU1MnIGlmIHJbJ2xlYWthZ2UnXVsncGFzc2VkJ10gZWxzZSAnRkFJTCd9KioiLCAiIiwKICAgICAgICAgICAgICAgICAgclsibGVha2FnZSJdWyJpbnRlcnByZXRhdGlvbiJdLCAiIl0KICAgICAgICAgICAgZm9yIGMgaW4gclsibGVha2FnZSJdWyJjaGVja3MiXToKICAgICAgICAgICAgICAgIEwuYXBwZW5kKGYiLSB7J09LJyBpZiBjWydwYXNzZWQnXSBlbHNlICdGQUlMJ30gYHtjWyduYW1lJ119YDoge2NbJ2RldGFpbCddfSIpCiAgICAgICAgICAgIEwuYXBwZW5kKCIiKQogICAgTCArPSBbIiMjIE1vZHVsZXMgMS00IGludGVncml0eSIsICIiLAogICAgICAgICAgZiJVbmNoYW5nZWQ6ICoqe3JlcG9ydFsnbW9kdWxlc18xXzRfaW50ZWdyaXR5J11bJ3VuY2hhbmdlZCddfSoqIiwgIiIsCiAgICAgICAgICAiIyMgV2FybmluZ3MiLCAiIl0gKyAoW2YiLSB7d30iIGZvciB3IGluIHJlcG9ydFsid2FybmluZ3MiXV0gb3IgWyItIG5vbmUiXSkKICAgIHBhdGgud3JpdGVfdGV4dCgiXG4iLmpvaW4oTCksIGVuY29kaW5nPSJ1dGYtOCIpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=", "sha256": "b8cda20db1827c304eb40c0fbeb6e2536684b9c1f24eb8b082ea2602d36341d8", "size": 12635}}'''
print('payload loaded:', len(PAYLOAD_JSON), 'chars')

In [ ]:
import base64, hashlib, json, os, shutil, subprocess, sys, time
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Fake_News_Detection")
assert PROJECT.exists(), f"Project not found: {PROJECT}. Mount Drive first."

PAYLOAD = json.loads(PAYLOAD_JSON)

# ---- 1. verify destination paths BEFORE writing anything ----
print("DESTINATION CHECK")
plan = []
for rel, meta in PAYLOAD.items():
    dest = PROJECT / rel
    assert dest.parent.exists(), f"Destination folder missing: {dest.parent}"
    action = "OVERWRITE" if dest.exists() else "CREATE"
    old = dest.stat().st_size if dest.exists() else 0
    plan.append((rel, action, old, meta["size"]))
    print(f"  {action:9} {rel}  ({old} -> {meta['size']} bytes)")

# ---- 2. back up anything being overwritten ----
stamp = time.strftime("%Y%m%d_%H%M%S")
backup = PROJECT / "backups" / f"module5_stage3_{stamp}"
n_backed = 0
for rel, action, _, _ in plan:
    if action == "OVERWRITE":
        b = backup / rel
        b.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(PROJECT / rel, b)
        n_backed += 1
print(f"\nBacked up {n_backed} file(s) -> {backup if n_backed else '(none needed)'}")

# ---- 3. write, then verify bytes by hash ----
print("\nWRITE + HASH VERIFY")
for rel, meta in PAYLOAD.items():
    dest = PROJECT / rel
    data = base64.b64decode(meta["b64"])
    dest.write_bytes(data)
    got = hashlib.sha256(dest.read_bytes()).hexdigest()
    ok = got == meta["sha256"] and dest.stat().st_size == meta["size"]
    print(f"  {'OK  ' if ok else 'FAIL'} {rel}  {dest.stat().st_size} bytes  sha256={got[:16]}...")
    assert ok, f"HASH MISMATCH after writing {rel}"

# ---- 4. clear stale bytecode so Python cannot import the old module ----
removed = 0
for d in ("src/fact_verification", "tests", "scripts"):
    pyc = PROJECT / d / "__pycache__"
    if pyc.exists():
        shutil.rmtree(pyc, ignore_errors=True); removed += 1
print(f"\nCleared {removed} stale __pycache__ dir(s)")

In [ ]:
import hashlib, re as _re, subprocess, sys
from pathlib import Path
PROJECT = Path("/content/drive/MyDrive/Fake_News_Detection")
FAILURES = []

def run(cmd):
    r = subprocess.run(cmd, cwd=PROJECT, shell=True, capture_output=True, text=True)
    return r.returncode, (r.stdout + r.stderr)

def require(name, ok, detail=""):
    print(f"  {'PASS' if ok else 'FAIL'}  {name}" + (f"  |  {detail}" if detail else ""))
    if not ok:
        FAILURES.append(f"{name}: {detail}")

print("=== 6. py_compile (implementation files) ===")
rc, out = run("python -m py_compile src/fact_verification/leakage_gate.py "
              "src/fact_verification/corpus_builder.py scripts/build_claim_corpus.py")
require("py_compile all three implementation files", rc == 0, out.strip()[:300] or "clean")

print("\n=== 5. content assertions ===")
lg = (PROJECT / "src/fact_verification/leakage_gate.py").read_text(encoding="utf-8")
cb = (PROJECT / "src/fact_verification/corpus_builder.py").read_text(encoding="utf-8")
tf = (PROJECT / "tests/test_fact_verification.py").read_text(encoding="utf-8")
require("leakage_gate defines candidate_pairs",        "def candidate_pairs(" in lg)
require("group_claims uses the optimized filter",      "for i, j in candidate_pairs(" in lg)
require("verify_no_leakage uses the optimized filter", lg.count("candidate_pairs(") >= 3)
require("old coarse blocking path removed",            "buckets.setdefault(bkey" not in lg)
require("corpus_builder uses SHA-256 evidence_id",     "hashlib.sha256(normalized_text.encode" in cb)
require("evidence_id is NOT a raw-text prefix",        "hash_key(e_norm)[:64]" not in cb)
require("build_claim_corpus.py exists",                (PROJECT / "scripts/build_claim_corpus.py").exists())
require("regression test: 64-char prefix collision",   "test_evidence_ids_differ_for_shared_64_char_prefix" in tf)
require("regression test: filter exactness",           "test_candidate_filter_is_exact_vs_brute_force" in tf)
require("regression test: row accounting",             "test_builder_accounts_for_every_input_row" in tf)

print("\n=== 5b. duplicate scan ===")
for pat, folder in [("build_claim_corpus*.py", "scripts"), ("corpus_builder*.py", "src/fact_verification"),
                    ("leakage_gate*.py", "src/fact_verification"), ("test_fact_verification*.py", "tests")]:
    hits = [q for q in (PROJECT / folder).glob(pat) if "__pycache__" not in str(q)]
    require(f"exactly one {folder}/{pat}", len(hits) == 1, f"found {[q.name for q in hits]}")

print("\n=== 7. test suites (executed against the DRIVE copies) ===")
EXPECTED = {"tests/test_fact_verification.py": 43, "tests/test_module5_corpus_schema.py": 21,
            "tests/test_contrastive.py": 22, "tests/test_embeddings.py": 5}
for t, exp in EXPECTED.items():
    rc, out = run(f"python {t}")
    m = None
    for line in out.strip().splitlines():
        hit = _re.search(r"(\d+)/(\d+) passed", line)
        if hit:
            m = hit
    if m is None:
        require(f"{t} produced a result line", False, out.strip()[-300:])
        continue
    got, tot = int(m.group(1)), int(m.group(2))
    require(f"{t} = {exp}/{exp}", rc == 0 and got == tot == exp, f"reported {got}/{tot}")

print("\n=== 8. Modules 1-4 integrity (read-only) ===")
EXPECT_MD5 = {"datasets/processed/train.csv": "fc26454e56cb9704509ffe3216d52a22",
              "datasets/processed/validation.csv": "081f7b828477ab9f27725d17c34b525a",
              "datasets/processed/test.csv": "66599a4f9d71bd5cee1927c3a573f0ce"}
for rel, exp in EXPECT_MD5.items():
    f = PROJECT / rel
    if not f.exists():
        require(f"{rel} present", False, "FILE MISSING"); continue
    h = hashlib.md5()
    with f.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    require(f"{rel} md5 unchanged", h.hexdigest() == exp, f"got {h.hexdigest()}")

print("\n" + "=" * 62)
if FAILURES:
    print(f"SYNC VERIFICATION FAILED - {len(FAILURES)} check(s):")
    for f in FAILURES:
        print(f"  - {f}")
    raise SystemExit("STOPPED: checks failed. Do NOT run the corpus build.")
print("ALL CHECKS PASSED.")
print("scripts/build_claim_corpus.py was NOT executed by this notebook.")
print("=" * 62)